# 02 Clean

Fills gaps in flight attributes from the history of the same flight number. Departures not matched to an NM flight
(no `FLIGHT_ID_mvt`, about 1 % of the flights) miss all `_flt` columns; most of them can be recovered this way.

The lookups are built from the 2025 training data only and applied unchanged to every dataset, so a flight gets the
same value whichever files are processed together. Labels are never touched.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
RAW = ROOT / "data/01_raw"
CLEAN = ROOT / "data/02_clean"

# Training files build the lookups; every file listed here is cleaned and written to data/02_clean.
TRAINING = ["train_DEP_merged_sorted.parquet", "train_ARR_merged_sorted.parquet"]
DATASETS = TRAINING + ["ranking.parquet", "final_ranking.parquet"]

In [2]:
data = {name: pd.read_parquet(RAW / name) for name in DATASETS if (RAW / name).is_file()}
missing_files = sorted(set(DATASETS) - set(data))
print("Not found, skipped:", missing_files or "none")
pd.DataFrame({name: [len(df), df["PHASE_mvt"].value_counts().to_dict()] for name, df in data.items()},
             index=["rows", "phases"]).T

Not found, skipped: none


,rows,phases
train_DEP_merged_sorted.parquet,2085047,{'DEP': 2085047}
train_ARR_merged_sorted.parquet,2082750,{'ARR': 2082750}
ranking.parquet,689534,"{'DEP': 344841, 'ARR': 344693}"
final_ranking.parquet,1341197,"{'DEP': 670790, 'ARR': 670407}"


## Helpers

`majority_lookup` keeps a flight number only if it has enough known values and one value clearly dominates.
`fill` writes the lookup into the missing cells and reports how many gaps it closed.

In [3]:
def training_values(columns):
    return pd.concat([data[name][columns] for name in TRAINING if name in data], ignore_index=True)


def majority_lookup(known, keys, column, min_entries, min_agreement):
    counts = known.dropna(subset=keys + [column]).groupby(keys + [column]).size().rename("count").reset_index()
    counts["total"] = counts.groupby(keys)["count"].transform("sum")
    best = counts.loc[counts.groupby(keys)["count"].idxmax()]
    best = best[(best["total"] >= min_entries) & (best["count"] / best["total"] >= min_agreement)]
    return best[keys + [column]]


def fill(column, lookup, keys=("FLIGHT_mvt",)):
    keys, rows = list(keys), []
    for name, df in data.items():
        missing = df[column].isna()
        values = df.loc[missing, keys].merge(lookup, on=keys, how="left", validate="many_to_one")[column]
        df.loc[missing, column] = values.to_numpy()
        rows.append({"dataset": name, "missing before": int(missing.sum()),
                     "filled": int(values.notna().sum()), "missing after": int(df[column].isna().sum())})
    return pd.DataFrame(rows).set_index("dataset")

## Arrival airport

At least 5 known flights with the same destination in 95 % of them.

In [4]:
ades = majority_lookup(training_values(["FLIGHT_mvt", "ADES_mvt"]), ["FLIGHT_mvt"], "ADES_mvt", 5, 0.95)
fill("ADES_mvt", ades)

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,122,2,120
train_ARR_merged_sorted.parquet,0,0,0
ranking.parquet,13,0,13
final_ranking.parquet,32,0,32


## Aircraft type

At least 30 known flights and 95 % agreement; aircraft types change more often than routes.

In [5]:
aircraft = majority_lookup(training_values(["FLIGHT_mvt", "AIRCRAFT_TYPE_mvt"]), ["FLIGHT_mvt"],
                           "AIRCRAFT_TYPE_mvt", 30, 0.95)
fill("AIRCRAFT_TYPE_mvt", aircraft)

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,1507,336,1171
train_ARR_merged_sorted.parquet,142,16,126
ranking.parquet,409,29,380
final_ranking.parquet,662,47,615


Types that are still unknown get a placeholder from the wake category known for the flight number:
`unknown_wtc_<cat>` for one category, `unknown_multi` for several, `unknown` otherwise.

In [6]:
wtc = training_values(["FLIGHT_mvt", "WK_TBL_CAT_flt"])
wtc["WK_TBL_CAT_flt"] = wtc["WK_TBL_CAT_flt"].astype("string").str.strip().replace("", pd.NA)
wtc = wtc.groupby("FLIGHT_mvt")["WK_TBL_CAT_flt"].agg(["nunique", "first"])
placeholder = ("unknown_wtc_" + wtc["first"]).mask(wtc["nunique"] > 1, "unknown_multi").fillna("unknown")

counts = {}
for name, df in data.items():
    missing = df["AIRCRAFT_TYPE_mvt"].isna()
    labels = df.loc[missing, "FLIGHT_mvt"].map(placeholder).fillna("unknown")
    df.loc[missing, "AIRCRAFT_TYPE_mvt"] = labels
    counts[name] = labels.value_counts()
pd.DataFrame(counts).fillna(0).astype(int)

,train_DEP_merged_sorted.parquet,train_ARR_merged_sorted.parquet,ranking.parquet,final_ranking.parquet
FLIGHT_mvt,,,,
unknown,194,79,229,365
unknown_multi,39,6,17,33
unknown_wtc_H,120,10,23,44
unknown_wtc_L,0,3,0,1
unknown_wtc_M,818,28,111,172


## Callsign

Not a model feature, but `03_adsb_fetch` uses it to find the aircraft in the ADS-B data.

In [7]:
callsign = majority_lookup(training_values(["FLIGHT_mvt", "CALLSIGN_flt"]), ["FLIGHT_mvt"], "CALLSIGN_flt", 10, 0.95)
fill("CALLSIGN_flt", callsign)

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,22470,15576,6894
train_ARR_merged_sorted.parquet,15449,10644,4805
ranking.parquet,7842,4589,3253
final_ranking.parquet,14129,8176,5953


## Market segment

First by flight number, then by flight number and aircraft type for the rest (both: 10 flights, 95 %).
Placeholder aircraft types are not used as a key.

In [8]:
market_known = training_values(["FLIGHT_mvt", "AIRCRAFT_TYPE_mvt", "MARKET_SEGMENT_flt"])
market = majority_lookup(market_known, ["FLIGHT_mvt"], "MARKET_SEGMENT_flt", 10, 0.95)
fill("MARKET_SEGMENT_flt", market)

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,22470,15348,7122
train_ARR_merged_sorted.parquet,15449,10980,4469
ranking.parquet,7842,4733,3109
final_ranking.parquet,14129,8453,5676


In [9]:
market_known = market_known[~market_known["AIRCRAFT_TYPE_mvt"].astype("string").str.startswith("unknown", na=False)]
market_by_type = majority_lookup(market_known, ["FLIGHT_mvt", "AIRCRAFT_TYPE_mvt"], "MARKET_SEGMENT_flt", 10, 0.95)
fill("MARKET_SEGMENT_flt", market_by_type, keys=("FLIGHT_mvt", "AIRCRAFT_TYPE_mvt"))

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,7122,1567,5555
train_ARR_merged_sorted.parquet,4469,48,4421
ranking.parquet,3109,379,2730
final_ranking.parquet,5676,565,5111


## Wake turbulence category

10 flights and 95 % agreement. Without a clear answer the value becomes `unknown_mix` (several categories seen)
or `unknown`, so the column has no gaps.

In [10]:
wake_known = training_values(["FLIGHT_mvt", "WK_TBL_CAT_flt"]).dropna()
wake_known = wake_known[~wake_known["WK_TBL_CAT_flt"].str.startswith("unknown")]
wake = majority_lookup(wake_known, ["FLIGHT_mvt"], "WK_TBL_CAT_flt", 10, 0.95).set_index("FLIGHT_mvt")["WK_TBL_CAT_flt"]
distinct = wake_known.groupby("FLIGHT_mvt")["WK_TBL_CAT_flt"].nunique()
wake_or_placeholder = wake.combine_first(pd.Series("unknown", index=distinct.index).mask(distinct > 1, "unknown_mix"))

rows = []
for name, df in data.items():
    missing = df["WK_TBL_CAT_flt"].isna()
    labels = df.loc[missing, "FLIGHT_mvt"].map(wake_or_placeholder).fillna("unknown")
    df.loc[missing, "WK_TBL_CAT_flt"] = labels
    rows.append({"dataset": name, "inferred": int((~labels.str.startswith("unknown")).sum()),
                 "unknown_mix": int((labels == "unknown_mix").sum()), "unknown": int((labels == "unknown").sum())})
pd.DataFrame(rows).set_index("dataset")

,inferred,unknown_mix,unknown
dataset,,,
train_DEP_merged_sorted.parquet,16553,1082,4909
train_ARR_merged_sorted.parquet,10489,911,4116
ranking.parquet,4823,430,2615
final_ranking.parquet,8463,806,4905


## Aircraft operator

10 flights and 95 % agreement.

In [11]:
operator = majority_lookup(training_values(["FLIGHT_mvt", "AIRCRAFT_OPERATOR_flt"]), ["FLIGHT_mvt"],
                          "AIRCRAFT_OPERATOR_flt", 10, 0.95)
fill("AIRCRAFT_OPERATOR_flt", operator)

,missing before,filled,missing after
dataset,,,
train_DEP_merged_sorted.parquet,22470,17499,4971
train_ARR_merged_sorted.parquet,15449,11313,4136
ranking.parquet,7842,5216,2626
final_ranking.parquet,14129,9199,4930


## Save

Same file names as in `01_raw`; the raw files stay untouched.

In [12]:
CLEAN.mkdir(parents=True, exist_ok=True)
for name, df in data.items():
    df.to_parquet(CLEAN / name, index=False)
    print(f"{name}: {len(df):,} rows")

train_DEP_merged_sorted.parquet: 2,085,047 rows


train_ARR_merged_sorted.parquet: 2,082,750 rows


ranking.parquet: 689,534 rows


final_ranking.parquet: 1,341,197 rows
